In [ ]:
import os
import pandas as pd
import re
import yaml  # pip install pyyaml

# === CONFIG ===
PROJECTS_DIR = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Config Files"
OUTPUT_DIR = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone\Analysis Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "7.1-YML_List_4th_attempt_ShallowC.csv")

# === UNIVERSAL UNIT TEST KEYWORDS ===
UNIT_TEST_KEYWORDS = [
    'gradlew test', './gradlew test', 'testdebugunittest', 'testreleaseunittest',
    'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

# === CI-SPECIFIC INSTRUMENTATION RULES ===
# (keep your detect_instrumentation_by_platform unchanged)

# === detect_ci_platform, detect_unit_test, extract_run_scripts unchanged ===

# === PROCESS ALL ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            # === NEW: Extract full_name between first and third '__'
            # Example: abcd__xyz__something__more.yml --> xyz__something
            parts = filename.split("__")
            if len(parts) >= 3:
                full_name = f"{parts[1]}__{parts[2]}"
            else:
                full_name = filename  # fallback if format is unexpected

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)

                    parsed = yaml.safe_load(raw)
                    run_scripts = extract_run_scripts(parsed)

                    is_unit = detect_unit_test(raw)
                    instr_types, matched_keys = detect_instrumentation_by_platform(ci_platform, raw, run_scripts, parsed)

                    test_type_str = ', '.join(sorted(instr_types))
                    is_instr = bool(instr_types)

                    results.append({
                        'filename': filename,
                        'full_name': full_name,
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': is_unit,
                        'instrumentation_test': is_instr,
                        'matched_keywords': "; ".join(sorted(set(matched_keys)))
                    })

                    print(f"✅ {filename} | CI: {ci_platform} | Unit: {is_unit} | Instr: {is_instr} | Keys: {matched_keys}")

            except Exception as e:
                print(f"❌ ERROR: {file_path} => {e}")
                results.append({
                    'filename': filename,
                    'full_name': full_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False,
                    'matched_keywords': ''
                })

# === EXPORT ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ DONE! CSV generated at: {OUTPUT_CSV}")


✅ 0000__JunkFood02__Seal__Other__FUNDING.yml | CI: Other | Unit: False | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__Other__bug_report.yml | CI: Other | Unit: True | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__Other__config.yml | CI: Other | Unit: False | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__Other__feature_request.yml | CI: Other | Unit: False | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__GitHub__Issue-Handler.yaml | CI: GitHub | Unit: True | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__GitHub__android.yml | CI: GitHub | Unit: True | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__GitHub__android_ci.yml | CI: GitHub | Unit: True | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__GitHub__close-stale-issues.yml | CI: GitHub | Unit: True | Instr: False | Keys: []
✅ 0000__JunkFood02__Seal__GitHub__sponsor.yml | CI: GitHub | Unit: True | Instr: False | Keys: []
✅ 0001__android__nowinandroid__Other__bug_report.yml | CI: Other | Unit: False | Instr: Fals